In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
# Notebook: PASSO 5 — Treinamento e comparação (Bagged Trees vs Weighted KNN)
# Entrada:  emodnet_pca_features  (PC1..PCk + QualityClass + chaves)
# Saídas:
#   1) emodnet_ml_scored                 -> predições dos dois modelos para todo o dataset
#   2) emodnet_ml_metrics_summary        -> métricas (val/test) por modelo (accuracy + macro F1 etc.)
#   3) emodnet_ml_confusion_matrices     -> matrizes de confusão (val/test) por modelo
#   4) emodnet_ml_models_artifacts       -> metadados do treino (split, params). (tabela)
#
# Observações:
# - Split estratificado: 70/15/15 (como no trabalho anterior)
# - Features: PC1..PCk (k vem do próprio dataset)
# - Label: QualityClass
# - Modelos:
#    * Bagged Trees = BaggingClassifier(estimator=DecisionTreeClassifier)
#    * Weighted KNN = KNeighborsClassifier(weights="distance")
#
# Nota importante:
# - Como estamos em Spark Notebook, vamos treinar via scikit-learn convertendo para pandas.
#   (16k linhas está ótimo para isso.)
# - Se futuramente crescer muito, migramos para Spark ML.

# PASSO 5 — ML Multitemporal (usando 3 PCs)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from pyspark.sql import functions as F
from pyspark.sql import types as T

# =========================
# CONFIG
# =========================
IN_TABLE = "emodnet_pca_features"

OUT_SCORED_TABLE  = "emodnet_ml_scored"
OUT_METRICS_TABLE = "emodnet_ml_metrics_summary"

RANDOM_STATE = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15

CLASS_ORDER = ["Excellent", "Good", "Fair", "Marginal", "Poor"]

N_ESTIMATORS = 200
KNN_K = 15

# =========================
# LOAD
# =========================
df = spark.table(IN_TABLE)

# Detecta PCs automaticamente
pc_cols = sorted(
    [c for c in df.columns if c.startswith("PC")],
    key=lambda x: int(x.replace("PC", ""))
)

print("PCs utilizados:", pc_cols)

required = ["QualityClass", "latitude", "longitude", "time"] + pc_cols
df = df.select(*required)

pdf = df.toPandas()

X = pdf[pc_cols].values
y = pdf["QualityClass"].values

# =========================
# SPLIT 70/15/15
# =========================
temp_size = TEST_SIZE + VAL_SIZE

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=temp_size,
    stratify=y,
    random_state=RANDOM_STATE
)

test_prop = TEST_SIZE / temp_size

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=test_prop,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

# =========================
# MODELOS
# =========================
bagged = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    n_estimators=N_ESTIMATORS,
    bootstrap=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

knn = KNeighborsClassifier(
    n_neighbors=KNN_K,
    weights="distance"
)

models = {
    "BaggedTrees": bagged,
    "WeightedKNN": knn
}

# =========================
# TREINO + MÉTRICAS
# =========================
metrics_rows = []

for name, model in models.items():
    model.fit(X_train, y_train)

    for split_name, X_eval, y_eval in [
        ("val", X_val, y_val),
        ("test", X_test, y_test)
    ]:
        y_pred = model.predict(X_eval)

        acc = accuracy_score(y_eval, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_eval, y_pred,
            average="macro",
            zero_division=0
        )

        print(f"\n{name} — {split_name.upper()}")
        print(f"Accuracy: {acc:.4f}")
        print(f"Macro F1: {f1:.4f}")

        metrics_rows.append((
            name,
            split_name,
            float(acc),
            float(prec),
            float(rec),
            float(f1)
        ))

# =========================
# SALVAR MÉTRICAS
# =========================
metrics_schema = T.StructType([
    T.StructField("model", T.StringType(), False),
    T.StructField("split", T.StringType(), False),
    T.StructField("accuracy", T.DoubleType(), False),
    T.StructField("macro_precision", T.DoubleType(), False),
    T.StructField("macro_recall", T.DoubleType(), False),
    T.StructField("macro_f1", T.DoubleType(), False),
])

metrics_df = spark.createDataFrame(metrics_rows, schema=metrics_schema) \
                  .withColumn("generated_at_utc", F.current_timestamp())

(
    metrics_df.write
      .mode("overwrite")
      .format("delta")
      .saveAsTable(OUT_METRICS_TABLE)
)

print(f"\n✅ Métricas salvas em: {OUT_METRICS_TABLE}")

# =========================
# PREDIÇÃO GLOBAL
# =========================
for name, model in models.items():
    pdf[f"Pred_{name}"] = model.predict(X)

sdf_scored = spark.createDataFrame(pdf)

(
    sdf_scored.write
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .format("delta")
      .saveAsTable(OUT_SCORED_TABLE)
)

print(f"✅ Predições salvas em: {OUT_SCORED_TABLE}")


StatementMeta(, e77d48d3-393c-4f31-9fb2-395a8eaeee96, 3, Finished, Available, Finished, False)

PCs utilizados: ['PC1', 'PC2', 'PC3']



BaggedTrees — VAL
Accuracy: 0.9828
Macro F1: 0.9827

BaggedTrees — TEST
Accuracy: 0.9799
Macro F1: 0.9799



WeightedKNN — VAL
Accuracy: 0.9836
Macro F1: 0.9836

WeightedKNN — TEST
Accuracy: 0.9791
Macro F1: 0.9790

✅ Métricas salvas em: emodnet_ml_metrics_summary
✅ Predições salvas em: emodnet_ml_scored
